# Create embeddings of documents

#### Setup Environment

In [1]:
import os
from dotenv import load_dotenv
from collections import defaultdict
from openai import OpenAI
from utils import save_defaultdict, load_defaultdict
import pandas as pd
from semantic_text_splitter import TextSplitter
from tokenizers import Tokenizer

In [2]:
# Loads variables from the environment
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")

#### Read in and Chunk Documents 

#### Create Embedding Vectors for Chunks

In [27]:
client = OpenAI(api_key=open_api_key)

max_tokens = 1023 # 8191 is max length for text-embedding-3-large
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")
splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, max_tokens)

# Create lists to store the data
embeddings = []
video_ids = []
chunk_indices = []
chunk_texts = []

# ToDo: add some sort of loop through all videos
documents = ['2Ds1m5gflCI', '77CdVSpnUX4']
for video_id in documents:

    with open(f'data/documents/{video_id}.txt', 'r', encoding='utf-8') as file:
        text_content = file.read()

    chunks = splitter.chunks(text_content)

    for chunk in chunks:
        # Create chunks directory if it doesn't exist
        os.makedirs('data/chunks', exist_ok=True)
        
        # Write chunk to file with video ID and chunk number
        chunk_filename = f'data/chunks/{video_id}_chunk{chunks.index(chunk)}.txt'
        with open(chunk_filename, 'w', encoding='utf-8') as f:
            f.write(chunk)
        response = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-large"
        )

        embeddings.append(response.data[0].embedding)
        video_ids.append(video_id)
        chunk_indices.append(chunks.index(chunk))
        # chunk_texts.append(chunk)


In [28]:
# Create DataFrame
df = pd.DataFrame({
    'embedding': embeddings,
    'video_id': video_ids, 
    'chunk_index': chunk_indices,
    # 'chunk_text': chunk_texts
})

In [29]:
df.head()

,embedding,video_id,chunk_index
0,"[0.0654204934835434, -0.06448127329349518, -0....",2Ds1m5gflCI,0
1,"[0.05524725466966629, -0.035123083740472794, -...",2Ds1m5gflCI,1
2,"[0.04196470230817795, -0.03418752923607826, -0...",2Ds1m5gflCI,2
3,"[0.06123918294906616, -0.013619143515825272, -...",2Ds1m5gflCI,3
4,"[0.06061168015003204, -0.041039951145648956, -...",2Ds1m5gflCI,4


In [30]:
# Save DataFrame to CSV
df.to_csv('data/embeddings.csv', index=False)

Potential design options:
1. create a list of dictionaries and store the data in a dataframe using columns: 'embedding', 'rank', 'chunk_id', 'video_id'
2. redis
3. vector db
4. mongo collections
5. pickle

ToDo:
- Add chunks to Mongo
- Make Query embedding function
- Rank Query to embedding with chunks
    - Will need to engineer a data object to handle this 
        - Add a secondary "key" object?
- Generate response using top chunks